# Witness test for math — does giving the answer actually help?

**Hypothesis.** Unlike debugging (see `PedRL_debug/`), math is *not* reversal-friendly: handing a
model the final answer (the *witness*) rarely converts an incapable model into a capable one,
because the hard part is constructing the derivation, not recognizing the target. If true, the
privileged/blind gap that Pedagogical RL wants to distill is small in math — matching the round-1
findings in `RESULTS.md`.

**Design** (same base model in both conditions, recent AIME 2024 + 2025, 60 problems):

1. **pass@1, blind vs hinted** — sample the model k times with the plain problem (*student*) and
   k times with the gold answer in the system prompt (*teacher*, `TEACHER_SYSTEM_ANSWER` from
   `pedrl/data.py`).
2. **Rescued set** — problems the student fails 0/k but the teacher solves ≥1/k, and the rescue
   ratio among student-failed problems. This is the raw "witness gap".
3. **LLM judge** — raw hinted pass@1 is inflated by *answer backfilling* (assert the given answer
   without a real derivation). Claude grades every rescuing solution: does it legitimately derive
   the answer, or work backwards from it?
4. **Surprisal heatmaps** — for the legitimate rescues, per-token surprise gaps
   `d_t = log p(argmax) − log p(actual)` scored under the blind context, and the *witness effect*
   `d_t(blind) − d_t(hinted)`: tokens that are only plausible **given** the answer light up.

The debugging mirror of steps 1–2 is `python PedRL_debug/run.py probe` — compare the two gaps.

> Runtime: ~30–60 min on an A100 with the 7B default (generation dominates). Use a 1.5B model on a T4.
> The judge step needs an `ANTHROPIC_API_KEY` (Colab: add it in the Secrets sidebar).

In [ ]:
%pip install -q -U "transformers>=4.49.0" "datasets>=2.19.0" "accelerate>=0.34.0" "anthropic>=0.40.0" matplotlib
!nvidia-smi | head -12

In [ ]:
# Get the repo (skipped when already inside it) and optional tokens
import os, sys

if os.path.basename(os.getcwd()) == "pedrl":
    os.chdir("..")                     # notebook opened from inside pedrl/ locally
if os.path.basename(os.getcwd()) != "PedRL" and not os.path.exists("pedrl"):
    if not os.path.exists("PedRL"):
        !git clone https://github.com/dannyyoon0303/PedRL.git
    os.chdir("PedRL")
sys.path.insert(0, os.getcwd())

try:
    from google.colab import userdata
    for key in ("HF_TOKEN", "ANTHROPIC_API_KEY"):
        try:
            os.environ.setdefault(key, userdata.get(key))
            print(key, "set")
        except Exception:
            print(key, "not configured (HF optional; ANTHROPIC needed for the judge step)")
except ImportError:
    pass
print("cwd:", os.getcwd())

In [ ]:
CFG = dict(
    # A100 default. T4 fallback: "Qwen/Qwen2.5-1.5B-Instruct" (expect near-zero pass@1 on AIME —
    # fine for the hypothesis, which is about the blind->hinted LIFT, not the absolute level).
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    k = 8,                    # samples per problem per condition
    temperature = 0.9,        # mirrors GRPO sampling in the main repo
    top_p = 0.95,
    max_new_tokens = 2048,    # AIME needs long chains of thought
    gen_batch_size = 8,       # sequences per generate() call (k=8 -> 1 prompt per call)
    n_problems = 60,          # 30 (AIME 2024) + 30 (AIME 2025); reduce for a quick pass
    seed = 42,
    out_dir = "outputs_witness_math",
    judge_model = "claude-opus-4-8",
    max_judged_per_problem = 2,   # teacher-correct samples judged per rescued problem
    heatmap_bins = 256,           # token axis of the heatmaps is max-pooled into this many bins
)
import os
os.makedirs(CFG["out_dir"], exist_ok=True)

## 1. Data — AIME 2024 + 2025

Both conditions reuse the prompt builders from `pedrl/data.py`, so the hinted prompt is exactly the
teacher prompt PedRL trains with (`privileged="answer"`). AIME answers are integers 0–999, so the
repo's `extract_prediction` / `answers_match` grading works unchanged.

In [ ]:
import json, re
from datasets import load_dataset
from pedrl.data import build_prompts
from pedrl.modeling import load_tokenizer

def _boxed_int(s):
    m = re.findall(r"\\boxed\{(\d+)\}", s or "")
    return m[-1] if m else None

problems = []
for name, year in [("math-ai/aime24", 2024), ("math-ai/aime25", 2025)]:
    for ex in load_dataset(name, split="test"):
        ans = ex.get("answer")
        ans = str(ans) if ans is not None else _boxed_int(ex.get("solution"))
        if ans is None:
            continue
        problems.append({"id": f"{year}-{ex['id']}", "year": year,
                         "problem": ex["problem"], "answer": ans})
problems = problems[:CFG["n_problems"]]

tokenizer = load_tokenizer(CFG["model_name"])
for p in problems:
    p["student_prompt"], p["teacher_prompt"] = build_prompts(
        tokenizer, p["problem"], p["answer"], "", privileged="answer")

print(f"{len(problems)} problems "
      f"({sum(p['year'] == 2024 for p in problems)} from 2024, "
      f"{sum(p['year'] == 2025 for p in problems)} from 2025)")
print("\n--- hinted (teacher) prompt example ---\n", problems[0]["teacher_prompt"][:700])

## 2. Generate — same model, blind vs hinted

Cached to `outputs_witness_math/generations.json`; delete that file to regenerate.

In [ ]:
import torch
from pedrl.modeling import load_model, batch_generate, set_seed

gen_path = os.path.join(CFG["out_dir"], "generations.json")
if os.path.exists(gen_path):
    with open(gen_path) as f:
        generations = json.load(f)
    print("loaded cached generations — delete the file to regenerate:", gen_path)
else:
    set_seed(CFG["seed"])
    model = load_model(CFG["model_name"])
    model.eval()
    generations = {}
    for cond, col in [("blind", "student_prompt"), ("hinted", "teacher_prompt")]:
        print(f"[{cond}] {len(problems)} problems x {CFG['k']} samples "
              f"(temp={CFG['temperature']})")
        groups = batch_generate(
            model, tokenizer, [p[col] for p in problems],
            max_new_tokens=CFG["max_new_tokens"], batch_size=CFG["gen_batch_size"],
            temperature=CFG["temperature"], top_p=CFG["top_p"],
            num_return_sequences=CFG["k"])
        generations[cond] = [[{"text": g["text"], "ids": g["ids"]} for g in grp]
                             for grp in groups]
    with open(gen_path, "w") as f:
        json.dump(generations, f)
    print("->", gen_path)

## 3. pass@1 and the rescued set

The number that tests the hypothesis is the **rescue ratio**: among problems the blind student
fails 0/k, what fraction does the answer-hinted teacher solve at least once? (In debugging we
expect this to be large; the math hypothesis says it stays small — and step 4 will shrink it
further by discarding backfills.)

In [ ]:
from pedrl.data import extract_prediction, answers_match

def n_correct(group, gold):
    return sum(answers_match(extract_prediction(g["text"]), gold) for g in group)

k, n = CFG["k"], len(problems)
stats = []
for i, p in enumerate(problems):
    stats.append({"id": p["id"], "year": p["year"], "answer": p["answer"],
                  "blind": n_correct(generations["blind"][i], p["answer"]),
                  "hinted": n_correct(generations["hinted"][i], p["answer"])})

pass1 = {c: sum(s[c] for s in stats) / (n * k) for c in ("blind", "hinted")}
failed_blind = [s for s in stats if s["blind"] == 0]
rescued = [s for s in stats if s["blind"] == 0 and s["hinted"] > 0]
anti = [s for s in stats if s["blind"] > 0 and s["hinted"] == 0]

print(f"pass@1: blind = {pass1['blind']:.3f}   hinted = {pass1['hinted']:.3f}   "
      f"lift = {pass1['hinted'] - pass1['blind']:+.3f}")
print(f"student fails 0/{k}:            {len(failed_blind)}/{n} problems")
print(f"rescued by the witness:        {len(rescued)}  "
      f"(rescue ratio = {len(rescued) / max(1, len(failed_blind)):.2f})")
print(f"anti-rescued (hint hurt):      {len(anti)}")

with open(os.path.join(CFG["out_dir"], "witness_math_results.json"), "w") as f:
    json.dump({"cfg": {k_: v for k_, v in CFG.items()}, "pass1": pass1,
               "n_failed_blind": len(failed_blind), "n_rescued": len(rescued),
               "per_problem": stats}, f, indent=2)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.2, 3))
vals = [pass1["blind"], pass1["hinted"]]
bars = ax.bar(["blind\n(student)", "answer hint\n(teacher)"], vals,
              color=["#9aa3af", "#4269d0"], width=0.55)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f" {v:.3f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("pass@1")
ax.set_title(f"AIME 24+25 — witness effect (raw, pre-judge)\n{CFG['model_name'].split('/')[-1]}, k={k}",
             fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 4. LLM judge — are the rescues legitimate?

A hinted model can emit `\boxed{answer}` without ever deriving it. Claude grades each rescuing
solution knowing the setup (the model was told the answer), into:

- **`legitimate`** — a complete derivation; every step justified; would convince a grader who does
  *not* know the answer.
- **`answer_backfill`** — works backwards from the given answer, verifies candidates against it, or
  asserts it without derivation.
- **`lucky_guess`** — the derivation is broken or irrelevant; the answer just appears at the end.
- **`flawed`** — a genuine attempt that reaches the answer through a significant logical gap or error.

Only `legitimate` counts as a real rescue — that's the honest witness gap for math.

In [ ]:
rescued_ids = {s["id"] for s in rescued}
to_judge = []
for i, p in enumerate(problems):
    if p["id"] not in rescued_ids:
        continue
    good = [g for g in generations["hinted"][i]
            if answers_match(extract_prediction(g["text"]), p["answer"])]
    for g in good[:CFG["max_judged_per_problem"]]:
        to_judge.append((p, g))
print(f"{len(to_judge)} rescuing solutions to judge "
      f"(from {len(rescued)} rescued problems)")

verdicts = []
if not to_judge:
    print("Nothing to judge: the witness rescued nothing — the hypothesis holds trivially here.")
else:
    import anthropic
    from pydantic import BaseModel
    from typing import Literal

    class Verdict(BaseModel):
        category: Literal["legitimate", "answer_backfill", "lucky_guess", "flawed"]
        legitimate: bool
        justification: str

    JUDGE_SYSTEM = (
        "You are a strict competition-mathematics grader. The solution you are given was written "
        "by a model that WAS TOLD the final answer in advance and instructed to solve the problem "
        "while landing on that answer. Your job is to decide whether the written solution "
        "legitimately DERIVES the answer, or merely reaches it because it was given.\n\n"
        "Grade `legitimate` only if the derivation is complete and each step is mathematically "
        "justified — it would convince a grader who does not know the answer. Grade "
        "`answer_backfill` if the solution works backwards from the stated answer, checks "
        "candidates against it, or asserts it without derivation. Grade `lucky_guess` if the "
        "reasoning is broken or irrelevant and the answer simply appears. Grade `flawed` if it is "
        "a genuine attempt that reaches the answer through a significant logical gap or error. "
        "Set legitimate=true only for the `legitimate` category. Be skeptical: unexplained "
        "constants, sudden simplifications toward the known target, and 'it can be shown that' "
        "leaps are hallmarks of backfilling.")

    client = anthropic.Anthropic()
    for p, g in to_judge:
        try:
            resp = client.messages.parse(
                model=CFG["judge_model"], max_tokens=1500,
                system=JUDGE_SYSTEM,
                messages=[{"role": "user", "content":
                           f"### Problem\n{p['problem']}\n\n### Gold answer\n{p['answer']}"
                           f"\n\n### Solution to grade\n{g['text']}"}],
                output_format=Verdict)
            v = resp.parsed_output
            verdicts.append({"id": p["id"], **v.model_dump(),
                             "completion": g["text"], "ids": g["ids"]})
            print(f"  {p['id']}: {v.category:15s} — {v.justification[:90]}")
        except Exception as e:
            print(f"  {p['id']}: judge error — {e!r}")

    legit = [v for v in verdicts if v["legitimate"]]
    legit_problems = {v["id"] for v in legit}
    print(f"\nlegitimate solutions: {len(legit)}/{len(verdicts)} judged   "
          f"legitimately rescued problems: {len(legit_problems)}/{len(rescued)}")
    print(f"HONEST rescue ratio (post-judge): "
          f"{len(legit_problems) / max(1, len(failed_blind)):.2f}   "
          f"(raw was {len(rescued) / max(1, len(failed_blind)):.2f})")
    with open(os.path.join(CFG["out_dir"], "judge_verdicts.json"), "w") as f:
        json.dump(verdicts, f, indent=2)

## 5. Surprisal heatmaps for the legitimate rescues

For each legitimate solution, per-token surprise gaps `d_t` under two contexts:

- **top (sequential, blue):** `d_t` under the **blind** student prompt — where the trajectory is
  off-policy for the student; spikes are the tokens PedRL's `G_spike` punishes.
- **bottom (diverging):** the **witness effect** `d_t(blind) − d_t(hinted)`. Red bins are tokens
  that only become plausible *given the answer* — the leak points a distillation student cannot
  reproduce blind. Blue bins are tokens the hint made *more* surprising (rare).

The token axis is max-pooled into fixed-width bins so spikes survive the compression.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

legit = [v for v in verdicts if v.get("legitimate")]
if not legit:
    print("No legitimate rescues to score — skip the heatmaps; the result stands on steps 3-4.")
else:
    if "model" not in globals():
        model = load_model(CFG["model_name"])
        model.eval()

    by_id = {p["id"]: p for p in problems}

    @torch.no_grad()
    def token_gaps(prompt, completion_ids):
        device = next(model.parameters()).device
        p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        seq = torch.tensor([p_ids + list(completion_ids)], device=device)
        logits = model(input_ids=seq).logits[0]
        sl = logits[len(p_ids) - 1 : len(p_ids) - 1 + len(completion_ids)].float()
        logp = torch.log_softmax(sl, dim=-1)
        tgt = torch.tensor(completion_ids, device=device)
        actual = logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
        return (logp.max(dim=-1).values - actual).cpu().numpy()

    rows_blind, rows_diff, labels = [], [], []
    for v in legit:
        p = by_id[v["id"]]
        gb = token_gaps(p["student_prompt"], v["ids"])
        gh = token_gaps(p["teacher_prompt"], v["ids"])
        rows_blind.append(gb)
        rows_diff.append(gb - gh)
        labels.append(f"{v['id']} ({len(gb)} tok)")
        print(f"{v['id']}: mean gap blind={gb.mean():.3f}  hinted={gh.mean():.3f}  "
              f"max witness effect={np.max(gb - gh):.2f} nats")

    def pool(x, bins, signed=False):
        idx = np.linspace(0, len(x), bins + 1).astype(int)
        out = np.zeros(bins)
        for j in range(bins):
            seg = x[idx[j]:idx[j + 1]]
            if len(seg):
                out[j] = seg[np.argmax(np.abs(seg))] if signed else seg.max()
        return out

    B = CFG["heatmap_bins"]
    H_blind = np.stack([pool(r, B) for r in rows_blind])
    H_diff = np.stack([pool(r, B, signed=True) for r in rows_diff])

    h = max(2.2, 0.42 * len(legit) + 1.4)
    fig, axes = plt.subplots(2, 1, figsize=(13, 2 * h), constrained_layout=True)

    im0 = axes[0].imshow(H_blind, aspect="auto", cmap="Blues", vmin=0, interpolation="nearest")
    axes[0].set_title("Surprise gap $d_t$ under the BLIND student  (max-pooled; darker = more off-policy)",
                      fontsize=10, loc="left")
    fig.colorbar(im0, ax=axes[0], label="$d_t$ (nats)", pad=0.01)

    vmax = np.percentile(np.abs(H_diff), 99) or 1.0
    im1 = axes[1].imshow(H_diff, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                         interpolation="nearest")
    axes[1].set_title("Witness effect  $d_t(\\mathrm{blind}) - d_t(\\mathrm{hinted})$   "
                      "(red = plausible only GIVEN the answer)", fontsize=10, loc="left")
    fig.colorbar(im1, ax=axes[1], label="$\\Delta d_t$ (nats)", pad=0.01)

    for ax in axes:
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_xlabel("token position (pooled bins)")
        for s in ax.spines.values():
            s.set_visible(False)
    plt.show()

In [ ]:
# Qualitative: the strongest leak points — tokens whose plausibility depends most on the answer
if legit:
    i_best = int(np.argmax([r.max() for r in rows_diff]))
    v, diff = legit[i_best], rows_diff[i_best]
    print(f"solution {v['id']} — top witness-dependent tokens (Δd_t = blind − hinted, in nats)\n")
    for t in sorted(np.argsort(diff)[::-1][:15]):
        ctx = tokenizer.decode(v["ids"][max(0, t - 10):t])
        tok = tokenizer.decode([v["ids"][t]])
        print(f"  pos {t:4d}  Δd_t={diff[t]:6.2f}   ...{ctx[-60:]!r} >>> {tok!r}")

## 6. Reading the result

**The hypothesis holds if:**

- the raw pass@1 lift from the hint is small, and the **honest (post-judge) rescue ratio** is near
  zero — most "rescues" are backfills, not derivations;
- the witness-effect heatmap shows red concentrated at *answer-adjacent* positions (the final
  `\boxed{}`, magnitude choices, "the answer is" spans) rather than spread through the reasoning —
  i.e. the hint changes *where the solution lands*, not *how the model reasons*.

**It fails if** the honest rescue ratio is substantial — then math *does* have exploitable witness
structure at this scale and the round-1 PedRL diagnosis needs revisiting.

**The contrast that motivates PedRL_debug:** run `python PedRL_debug/run.py probe --preset poc` on
the same GPU. Debugging's witness (the bug explanation) should produce a far larger honest gap —
that gap is the signal pedagogical distillation can transfer, and its absence here is why the math
PoC stalled in the dense regime.

**Caveats.** AIME 2024 may partially sit in the model's pretraining data (inflating *blind*
pass@1 — compare the 2024 vs 2025 rows in `witness_math_results.json`); k=8 gives coarse
per-problem estimates; one model family. The judge itself can be fooled by a sufficiently smooth
backfill — spot-check a few `legitimate` verdicts by hand.